<a href="https://colab.research.google.com/github/mohammadanas7777/Exploratory-Data-Analysis-projects/blob/main/Assignment_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Telecom churn prediction**    -



##### **Project Type**    - Unsupervised Clustering
##### **Contribution**    - Individual
##### **Contributor -** Mohammad Anas


# **Project Summary -**

# 📊 Telecom Customer Segmentation using ML (Clustering)
**Objective:** Segment telecom customers to extract meaningful groups and recommend marketing strategies.

**Dataset:** Telecom customer churn dataset.

**Method:** K-Means Clustering with PCA visualization.


# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


Telecom companies face significant revenue loss due to customer churn — when customers discontinue their subscription or switch to competitors. Understanding customer behavior and identifying distinct customer segments can help the company design targeted retention strategies and marketing campaigns.

The objective of this assignment is to:

Segment telecom customers into homogeneous groups based on usage patterns, service subscriptions, and account attributes.

Analyze the characteristics of each segment, including tenure, monthly charges, total services subscribed, and churn rate.

Provide actionable business insights and recommendations to reduce churn and increase customer loyalty.

**Type of ML task:**

Primary task: Unsupervised learning (K-Means clustering for customer segmentation)

Secondary analysis: Churn behavior analysis post-segmentation

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set(style="whitegrid")


### Dataset Loading

In [ ]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')
filepath_1 = '/content/drive/MyDrive/Colab Notebooks/Assignment_ML/WA_Fn-UseC_-Telco-Customer-Churn.csv'
telecomp = pd.read_csv(filepath_1)
df = telecomp.copy()

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

In [ ]:
columns = df.columns

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
df.duplicated().sum()

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
df.isnull().sum()

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False)

### What did you know about your dataset?

* The dataset contains 7043 non-null values.

* The restaurant dataset contains information about 'customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'.






## ***2. Understanding Your Variables***

In [ ]:
# Dataset Describe
df.describe(include='all')

In [ ]:
# Converting 'TotalCharges' column from object type to float type
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for i in df.columns.tolist():
  print("No. of unique values in ",i,"is",df[i].nunique(),".")

In [ ]:
cat_columns = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
cont_columns = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
y = df['Churn']

### 📦 Boxplots to Visualize Outliers
Run boxplots for numeric columns (tenure, MonthlyCharges, TotalCharges).


In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
plt.figure(figsize=(12,4))
for i, col in enumerate(numeric_cols):
    plt.subplot(1,3,i+1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.show()


In [ ]:
### ⚙️ Cap Outliers (IQR Method)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5*IQR, Q3 + 1.5*IQR)


In [ ]:
# Service-related columns
service_cols = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]
df[service_cols] = df[service_cols].replace(
    {'Yes': 1, 'No': 0, 'No internet service': 0}
)

## 🛠 Encode Binary and Categorical Columns


In [ ]:
binary_map = {'Yes':1, 'No':0}
df['Partner'] = df['Partner'].map(binary_map)
df['Dependents'] = df['Dependents'].map(binary_map)
df['PhoneService'] = df['PhoneService'].map(binary_map)
df['PaperlessBilling'] = df['PaperlessBilling'].map(binary_map)
df['Churn'] = df['Churn'].map(binary_map)

df[service_cols] = df[service_cols].replace({'Yes':1,'No':0,'No internet service':0})


In [ ]:
# Ensure service columns are numeric (int)
df[service_cols] = df[service_cols].astype(int)

In [ ]:
df['TotalServices'] = df[service_cols].sum(axis=1)
contract_map = {'Month-to-month':0,'One year':1,'Two year':2}
df['Contract'] = df['Contract'].map(contract_map)
payment_map = {
    'Electronic check':0,'Mailed check':1,
    'Bank transfer (automatic)':2,'Credit card (automatic)':3
}
df['PaymentMethod'] = df['PaymentMethod'].map(payment_map)
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0,12,24,48,72], labels=[0,1,2,3])


## 📌 Standardize Features & Find Optimal Clusters


In [ ]:
features = ['tenure','MonthlyCharges','TotalCharges','TotalServices','Contract','PaymentMethod']
X = df[features]

In [ ]:
X.isnull().sum()

In [ ]:
X = X.apply(pd.to_numeric, errors='coerce')

In [ ]:
X = X.fillna(X.median())

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 🔎 Elbow Method


In [ ]:
wcss = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.plot(range(2,11), wcss, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.title('Elbow Method')
plt.show()

### 🪩 Silhouette Scores


In [ ]:
sil_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(6,4))
plt.plot(range(2,11), sil_scores, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

## 🕶 PCA to Visualize Clusters


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df['PCA1'] = X_pca[:,0]
df['PCA2'] = X_pca[:,1]

plt.figure(figsize=(8,6))
sns.scatterplot(
    x='PCA1',
    y='PCA2',
    hue='Cluster',
    palette='Set2',
    data=df
)
plt.title('Customer Segments (PCA)')
plt.show()

## 📌 Cluster Summary & Insights


In [ ]:
analysis_cols = features + ['Churn']

df[analysis_cols] = df[analysis_cols].apply(
    pd.to_numeric, errors='coerce'
)

In [ ]:
cluster_summary = df.groupby('Cluster')[analysis_cols].mean()
cluster_summary

In [ ]:
# Compute mean churn rate per cluster
churn_by_cluster = df.groupby('Cluster')['Churn'].mean().reset_index()

# Plot
plt.figure(figsize=(6,4))
sns.barplot(x='Cluster', y='Churn', data=churn_by_cluster, palette='Set2')
plt.title('Churn Rate by Customer Segment')
plt.ylabel('Average Churn Rate')
plt.xlabel('Customer Cluster')
plt.ylim(0,1)  # Optional: scale 0 to 1
plt.show()

Cluster 0: Long tenure, low charges, low churn → Loyal customers  
Cluster 1: High charges, month-to-month contracts → High churn risk  
Cluster 2: New customers, low service usage → Early churn risk  
Cluster 3: High service usage, medium tenure → Upsell opportunity

## Marketing Recommendations

• Provide loyalty discounts to high-churn segments  
• Promote long-term contracts to reduce churn  
• Upsell bundled services to high-usage customers  
• Reward loyal customers with referrals and incentives


## 💡 Actionable Recommendations

* Offer loyalty discounts to high-churn clusters
* Promote long-term contracts to stabilize revenue  
* Tailored service bundles for high-service clusters  
